# **RIDGE regression**

Marek Šugár

In this notebook we aim to optimize regularization parameter $\lambda$ for every single stock ticker included in database. Using this we can optimize the overall performance more however, needs to be strongly checked on testing as we are balancing on the edge of overfit, of course.

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from DataFramePrep import generate_TrainingDataFrame

# Metrics to measure successfulness
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler

# ML Stuff
from sklearn.linear_model import Ridge

# In case of convergence problem, supress warning
import warnings

#warnings.filterwarnings("ignore")

In [2]:
TrainingDataFrame, tickers, historic_columns, StockDataDatabase = generate_TrainingDataFrame()

# **RIDGE regression**

In [11]:
performance_tracker_regularization = {}

for ticker in tickers["Ticker"]:
    performance_tracker_regularization[ticker] = []
    for Alpha in range(120, 261, 10):
        Stock_Data = pd.read_sql(
            f"SELECT * FROM StockData WHERE Ticker='{ticker}' AND Date>='2017-09-14'",
            con=StockDataDatabase, parse_dates=["Date"]
        )
            
        Stock_Data = pd.merge(Stock_Data, TrainingDataFrame, on="Date")
        Stock_Data["Target"] = Stock_Data["Close"]
        Stock_Data = Stock_Data.dropna().reset_index(drop=True)

        training_length = 4
        prediction_length = 1

        MAPEs = []

        for window_start in range(len(Stock_Data) - (training_length + prediction_length)):
            # Define feature and target windows
            Training_Features = Stock_Data[historic_columns].iloc[window_start:window_start+training_length]
            Training_Target = Stock_Data["Target"].iloc[window_start:window_start+training_length]

            Test_Features = Stock_Data[historic_columns].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]
            Test_Target = Stock_Data["Target"].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]

            # Skip empty slices
            if Training_Features.empty or Test_Features.empty:
                continue
            
            # Scale only selected features
            scaler = StandardScaler()
            Training_Features = scaler.fit_transform(Training_Features)
            Test_Features = scaler.transform(Test_Features)

            # Fit KNN
            MODEL = Ridge(alpha=Alpha)
            MODEL.fit(Training_Features, Training_Target)

            # Predict and evaluate
            prediction = MODEL.predict(Test_Features)
            MAPEs.append(100 * mean_absolute_percentage_error(Test_Target, prediction))
            
        # Save results
        performance_tracker_regularization[ticker].append((Alpha, np.mean(MAPEs)))
        print(Alpha, np.mean(MAPEs))


120 1.2362928792044026
130 1.235717184580519
140 1.2351887067570348
150 1.2347345522933701
160 1.2344788814696095
170 1.2343219905492795
180 1.2343273193480884
190 1.2344427436866687
200 1.2346163513107138
210 1.2348543276173685
220 1.2351633727057547
230 1.2355504364829757
240 1.2360044168207354
250 1.236489892493427
260 1.2370223728825345
120 1.6631792279790745
130 1.6624793473421988
140 1.6620039000794677
150 1.661669598281136
160 1.6615339957662876
170 1.6615376616567565
180 1.6617028560919909
190 1.6619631357531435
200 1.6623327892645539
210 1.6627865693904795
220 1.663304086799076
230 1.6638552001810234
240 1.664447060496662
250 1.6651309917180692
260 1.665940411344459
120 2.6210153956465096
130 2.6205212765214903
140 2.6203238858728732
150 2.620300268028744
160 2.62064886585293
170 2.621258751501057
180 2.6220751432597735
190 2.623042964199428
200 2.624103320373116
210 2.6252919149011698
220 2.6266572722813946
230 2.6281295242352063
240 2.629714186466764
250 2.631349474353266
26

In [15]:
parameters = performance_tracker_regularization.copy()
optimal = {}

for ticker in tickers["Ticker"]:
    best_ones = sorted(parameters[ticker], key=lambda x: x[1])
    print(ticker, best_ones[0][0])

ACN 170
ADBE 160
AMD 150
AKAM 120
APH 190
ADI 260
AAPL 180
AMAT 230
ANET 140
ADSK 180
AVGO 200
CDNS 190
CDW 190
CSCO 180
CTSH 210
GLW 190
DELL 120
ENPH 130
EPAM 160
FFIV 160
FICO 240
FSLR 140
FTNT 170
IT 130
GEN 130
GDDY 150
HPE 140
HPQ 180
IBM 130
INTC 160
INTU 180
JBL 150
KEYS 220
KLAC 260
LRCX 230
MCHP 260
MU 190
MSFT 250
MPWR 260
MSI 230
NTAP 150
NVDA 180
NXPI 250
ON 220
ORCL 130
PANW 160
PTC 160
QCOM 260
ROP 210
CRM 140
STX 140
NOW 180
SWKS 260
SMCI 120
SNPS 180
TEL 190
TDY 180
TER 230
TXN 260
TRMB 150
TYL 200
VRSN 180
WDC 150
WDAY 150
ZBRA 180
